In [2]:
import pandas as pd

## Load The Dataset

In [5]:
sd = pd.read_csv('transactions.csv')
sub = pd.read_csv('subscriptions.csv')

In [6]:
sd = pd.read_csv("transactions.csv")
sub = pd.read_csv("subscriptions.csv")

print("Transactions Shape:", sd.shape)
print("Subscriptions Shape:", sub.shape)

Transactions Shape: (213341, 27)
Subscriptions Shape: (674, 12)


In [7]:
# Dataset Columns
print("Transactions Columns:")
print(sd.columns.tolist())

print("\nSubscriptions Columns:")
print(sub.columns.tolist())

# Missing Values
print("\nTransactions Missing Values:")
print(sd.isnull().sum()[sd.isnull().sum() > 0])

print("\nSubscriptions Missing Values:")
print(sub.isnull().sum()[sub.isnull().sum() > 0])

# Duplicate Rows
print("\nTransactions Duplicates:", sd.duplicated().sum())
print("Subscriptions Duplicates:", sub.duplicated().sum())

Transactions Columns:
['transaction_id', 'account_id', 'customer_id', 'timestamp', 'amount', 'currency', 'merchant_name', 'merchant_category', 'merchant_mcc', 'transaction_type', 'payment_channel', 'city', 'country', 'counterparty_iban', 'label', 'category', 'subcategory', 'is_subscription', 'is_salary', 'is_transfer', 'is_cash_withdrawal', 'balance_after_transaction', 'is_foreign', 'original_amount', 'original_currency', 'fx_rate', 'foreign_fee']

Subscriptions Columns:
['subscription_id', 'customer_id', 'account_id', 'merchant_name', 'category', 'subcategory', 'amount', 'frequency_months', 'billing_day', 'start_date', 'end_date', 'status']

Transactions Missing Values:
city                  99110
counterparty_iban    209343
dtype: int64

Subscriptions Missing Values:
end_date    564
dtype: int64

Transactions Duplicates: 0
Subscriptions Duplicates: 0


## Transactions Dataset Cleaning

In [8]:
sd.drop(
    columns=['city', 'counterparty_iban'],
    inplace=True
)

print("Dataset Shape:", sd.shape)
print("\nMissing Values:")
print(sd.isnull().sum().sum())

Dataset Shape: (213341, 25)

Missing Values:
0


## Subscriptions Dataset Check

In [9]:
print("Subscription Status:")
print(sub['status'].value_counts())

print("\nEnd Date Missing by Status:")
print(
    pd.crosstab(
        sub['status'],
        sub['end_date'].isnull(),
        margins=True
    )
)

Subscription Status:
status
active     572
churned    102
Name: count, dtype: int64

End Date Missing by Status:
end_date  False  True  All
status                    
active        8   564  572
churned     102     0  102
All         110   564  674


# Investigate 8 Active Subscriptions

In [10]:
active_end = sub[
    (sub['status'] == 'active') &
    (sub['end_date'].notna())
]

print(active_end[
    ['subscription_id', 'merchant_name',
     'start_date', 'end_date', 'status']
].to_string(index=False))

subscription_id    merchant_name start_date   end_date status
      S00000221 Childcare Center 2025-03-16 2026-10-14 active
      S00000329 Childcare Center 2024-04-30 2026-10-14 active
      S00000477 Childcare Center 2025-11-25 2026-10-14 active
      S00000553 Childcare Center 2024-04-25 2026-10-14 active
      S00000561 Childcare Center 2026-05-23 2026-10-14 active
      S00000601 Childcare Center 2024-12-02 2026-10-14 active
      S00000628 Childcare Center 2025-03-20 2026-10-14 active
      S00000642 Childcare Center 2026-08-26 2026-10-14 active


## Target Variable Analysis

In [11]:
print("Total Transactions:", len(sd))

print("\nSubscription Distribution:")
print(sd['is_subscription'].value_counts())

print("\nSubscription Percentage:")
print(
    sd['is_subscription'].value_counts(normalize=True) * 100
)

print("\nUnique Customers:")
print(sd['customer_id'].nunique())

print("\nUnique Merchants:")
print(sd['merchant_name'].nunique())

Total Transactions: 213341

Subscription Distribution:
is_subscription
False    205997
True       7344
Name: count, dtype: int64

Subscription Percentage:
is_subscription
False    96.557624
True      3.442376
Name: proportion, dtype: float64

Unique Customers:
100

Unique Merchants:
180


## Subscription Merchant Analysis

In [12]:
# Subscription Transactions
subs = sd[sd['is_subscription'] == True].copy()

# Top 15 Subscription Merchants
print("Top 15 Subscription Merchants:")

print(
    subs['merchant_name']
    .value_counts()
    .head(15)
)

# Customer-Merchant Recurring Groups
groups = subs.groupby(
    ['customer_id', 'merchant_name']
).size()

print("\nTotal Customer-Merchant Groups:", len(groups))

print("\nGroups with 1 Payment:", (groups == 1).sum())

print("Groups with 2+ Payments:", (groups >= 2).sum())

print("Groups with 3+ Payments:", (groups >= 3).sum())

print("\nMaximum Payments in One Group:", groups.max())

Top 15 Subscription Merchants:
merchant_name
ConEdison        643
Verizon          579
AT&T             576
Amazon Prime     517
National Grid    489
State Farm       485
Geico Auto       449
Xfinity          434
T-Mobile         429
Netflix          388
Disney+          364
Spotify          362
Blue Cross       358
Apple iCloud+    304
Hulu             207
Name: count, dtype: int64

Total Customer-Merchant Groups: 518

Groups with 1 Payment: 42
Groups with 2+ Payments: 476
Groups with 3+ Payments: 442

Maximum Payments in One Group: 36


## Recurring Payment Interval Analysis

In [13]:
# Convert timestamp
sd['timestamp'] = pd.to_datetime(sd['timestamp'])

# Select subscription transactions
subs = sd[sd['is_subscription'] == True].copy()

# Sort transactions
subs = subs.sort_values(
    ['customer_id', 'account_id', 'merchant_name', 'timestamp']
)

# Calculate payment intervals
subs['payment_interval'] = (
    subs.groupby(
        ['customer_id', 'account_id', 'merchant_name']
    )['timestamp']
    .diff()
    .dt.total_seconds() / 86400
)

# Remove missing intervals
intervals = subs['payment_interval'].dropna()

print("Total Valid Intervals:", len(intervals))

print("\nAverage Interval:", round(intervals.mean(), 2))
print("Median Interval:", round(intervals.median(), 2))

print("\nPayment Frequency Analysis:")

print("Weekly (6-8 days):",
      intervals.between(6, 8).sum())

print("Monthly (27-32 days):",
      intervals.between(27, 32).sum())

print("Quarterly (85-95 days):",
      intervals.between(85, 95).sum())

print("Yearly (350-380 days):",
      intervals.between(350, 380).sum())


Total Valid Intervals: 6825

Average Interval: 37.57
Median Interval: 30.81

Payment Frequency Analysis:
Weekly (6-8 days): 0
Monthly (27-32 days): 6100
Quarterly (85-95 days): 213
Yearly (350-380 days): 5


## Historical Features

In [14]:
# Sort transactions
sd = sd.sort_values(
    ['customer_id', 'account_id', 'merchant_name', 'timestamp']
).reset_index(drop=True)

# Group transactions
group = sd.groupby(
    ['customer_id', 'account_id', 'merchant_name']
)

# Previous payment date
sd['previous_payment_date'] = group['timestamp'].shift(1)

# Days between payments
sd['days_since_last_payment'] = (
    sd['timestamp'] - sd['previous_payment_date']
).dt.total_seconds() / 86400

# Number of previous payments
sd['prior_merchant_payments'] = group.cumcount()

# Previous payment amount
sd['previous_amount'] = group['amount'].shift(1)

# Amount difference
sd['amount_difference'] = (
    sd['amount'].abs() - sd['previous_amount'].abs()
).abs()

# Check new features
print(sd[[
    'merchant_name',
    'days_since_last_payment',
    'prior_merchant_payments',
    'amount_difference',
    'is_subscription'
]].head(15))

   merchant_name  days_since_last_payment  prior_merchant_payments  \
0       7-Eleven                      NaN                        0   
1       7-Eleven                 6.657199                        1   
2       7-Eleven                 8.822766                        2   
3       7-Eleven                 4.089051                        3   
4       7-Eleven                12.198819                        4   
5       7-Eleven                 6.915880                        5   
6       7-Eleven                10.722488                        6   
7       7-Eleven                21.931343                        7   
8       7-Eleven                 0.393600                        8   
9       7-Eleven                 2.747859                        9   
10      7-Eleven                 0.894641                       10   
11      7-Eleven                25.402488                       11   
12      7-Eleven                 3.900613                       12   
13      7-Eleven    

## Subscription vs Non-Subscription Feature Comparison

In [ ]:
# Compare Subscription vs Normal Transactions

features = [
    'days_since_last_payment',
    'prior_merchant_payments',
    'amount_difference'
]

print("Feature Comparison:")

print(sd.groupby('is_subscription')[features].median().round(2))

# Monthly payment pattern
sd['monthly_pattern'] = (sd['days_since_last_payment'].between(27, 32))
# Similar payment amountsd['similar_amount'] = (sd['amount_difference'] <= 2)
print("\nMonthly Pattern Percentage:")
print(sd.groupby('is_subscription')['monthly_pattern'].mean() * 100)
print("\nSimilar Amount Percentage:")
print(sd.groupby('is_subscription')['similar_amount'].mean() * 100)

Feature Comparison:
                 days_since_last_payment  prior_merchant_payments  \
is_subscription                                                     
False                              15.06                     20.0   
True                               30.74                     10.0   

                 amount_difference  
is_subscription                     
False                        13.49  
True                          0.00  

Monthly Pattern Percentage:
is_subscription
False    10.447725
True     77.355664
Name: monthly_pattern, dtype: float64

Similar Amount Percentage:
is_subscription
False    16.131788
True     85.049020
Name: similar_amount, dtype: float64


## Combined Pattern Analysis

In [17]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

# Combined rule
sd['rule_subscription'] = (sd['monthly_pattern'] &sd['similar_amount'] &(sd['prior_merchant_payments'] >= 1))
# Compare with actual labels
print("Combined Pattern Distribution:")

print(pd.crosstab(sd['is_subscription'],sd['rule_subscription'],margins=True))

# Evaluate rule-based detection
print("\nClassification Report:")

print(classification_report(sd['is_subscription'],sd['rule_subscription'],zero_division=0))

print("\nConfusion Matrix:")

print(confusion_matrix( sd['is_subscription'], sd['rule_subscription']))

Combined Pattern Distribution:
rule_subscription   False  True     All
is_subscription                        
False              202137  3860  205997
True                 1886  5458    7344
All                204023  9318  213341

Classification Report:
              precision    recall  f1-score   support

       False       0.99      0.98      0.99    205997
        True       0.59      0.74      0.66      7344

    accuracy                           0.97    213341
   macro avg       0.79      0.86      0.82    213341
weighted avg       0.98      0.97      0.97    213341


Confusion Matrix:
[[202137   3860]
 [  1886   5458]]


## Payment Interval Consistency

In [18]:
# Previous payment interval
group = sd.groupby(
    ['customer_id', 'account_id', 'merchant_name']
)

sd['previous_interval'] = (
    group['days_since_last_payment'].shift(1)
)

# Difference between consecutive intervals
sd['interval_consistency'] = (
    sd['days_since_last_payment'] -
    sd['previous_interval']
).abs()

# Quarterly recurring pattern
sd['quarterly_pattern'] = (
    sd['days_since_last_payment'].between(85, 95)
)

# Yearly recurring pattern
sd['yearly_pattern'] = (
    sd['days_since_last_payment'].between(350, 380)
)

# Compare features
print("Interval Consistency Comparison:")

print(
    sd.groupby('is_subscription')[
        ['interval_consistency', 'previous_interval']
    ].median().round(2)
)

print("\nQuarterly Pattern Percentage:")

print(
    sd.groupby('is_subscription')['quarterly_pattern']
    .mean() * 100
)

print("\nYearly Pattern Percentage:")

print(
    sd.groupby('is_subscription')['yearly_pattern']
    .mean() * 100
)

Interval Consistency Comparison:
                 interval_consistency  previous_interval
is_subscription                                         
False                           14.25              14.76
True                             1.38              30.67

Quarterly Pattern Percentage:
is_subscription
False    1.762647
True     2.709695
Name: quarterly_pattern, dtype: float64

Yearly Pattern Percentage:
is_subscription
False    0.065535
True     0.068083
Name: yearly_pattern, dtype: float64


## Target Leakage Check

In [20]:
check_cols = ['label', 'category', 'subcategory']

for col in check_cols:

    print(f"\n===== {col.upper()} =====")

    result = pd.crosstab(sd[col],sd['is_subscription'])

    result['Total'] = result.sum(axis=1)

    result['Subscription_Rate'] = (result.get(True, 0) / result['Total'] * 100)

    print(result.sort_values('Subscription_Rate',ascending=False).head(15))


===== LABEL =====
is_subscription   False  True  Total  Subscription_Rate
label                                                  
CHILDCARE CENTER      0   111    111         100.000000
BLUE CROSS            0   358    358         100.000000
STATE FARM            0   485    485         100.000000
T-MOBILE              0   429    429         100.000000
CITY WATER DEPT       0   127    127         100.000000
CONEDISON             0   643    643         100.000000
GEICO AUTO            0   449    449         100.000000
GITHUB                0    30     30         100.000000
GOOGLE WORKSPACE      0    83     83         100.000000
NATIONAL GRID         0   489    489         100.000000
XFINITY               0   434    434         100.000000
VERIZON               0   579    579         100.000000
ADOBE                 0    84     84         100.000000
AT&T                  0   576    576         100.000000
DISNEY+             499   364    863          42.178447

===== CATEGORY =====
is_subs

## Final Feature Selection

In [21]:
# Selected features
features = [
    'amount',
    'days_since_last_payment',
    'prior_merchant_payments',
    'amount_difference',
    'previous_interval',
    'interval_consistency',
    'monthly_pattern',
    'similar_amount',
    'quarterly_pattern',
    'yearly_pattern'
]

# Create ML dataset
model_df = sd[['timestamp', 'is_subscription'] + features].copy()

# Convert amount to positive magnitude
model_df['amount'] = model_df['amount'].abs()

# Convert boolean features to integers
bool_cols = [
    'monthly_pattern',
    'similar_amount',
    'quarterly_pattern',
    'yearly_pattern'
]

model_df[bool_cols] = model_df[bool_cols].astype(int)

# Handle missing historical features
missing_cols = [
    'days_since_last_payment',
    'amount_difference',
    'previous_interval',
    'interval_consistency'
]

model_df[missing_cols] = model_df[missing_cols].fillna(-1)

# Check final ML dataset
print("ML Dataset Shape:", model_df.shape)

print("\nMissing Values:")
print(model_df.isnull().sum())

print("\nTarget Distribution:")
print(model_df['is_subscription'].value_counts())

print("\nSelected Features:")
print(features)

ML Dataset Shape: (213341, 12)

Missing Values:
timestamp                  0
is_subscription            0
amount                     0
days_since_last_payment    0
prior_merchant_payments    0
amount_difference          0
previous_interval          0
interval_consistency       0
monthly_pattern            0
similar_amount             0
quarterly_pattern          0
yearly_pattern             0
dtype: int64

Target Distribution:
is_subscription
False    205997
True       7344
Name: count, dtype: int64

Selected Features:
['amount', 'days_since_last_payment', 'prior_merchant_payments', 'amount_difference', 'previous_interval', 'interval_consistency', 'monthly_pattern', 'similar_amount', 'quarterly_pattern', 'yearly_pattern']


## Time-Based Data Split

In [22]:
# Sort by time
model_df['timestamp'] = pd.to_datetime(model_df['timestamp'])

model_df = model_df.sort_values(
    'timestamp'
).reset_index(drop=True)

# Split sizes
train_size = int(len(model_df) * 0.70)
val_size = int(len(model_df) * 0.20)

# Time-based split
train_df = model_df.iloc[:train_size]
val_df = model_df.iloc[train_size:train_size + val_size]
test_df = model_df.iloc[train_size + val_size:]

# Features
X_train = train_df[features]
X_val = val_df[features]
X_test = test_df[features]

# Target
y_train = train_df['is_subscription'].astype(int)
y_val = val_df['is_subscription'].astype(int)
y_test = test_df['is_subscription'].astype(int)

# Check shapes
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

# Check class distribution
print("\nTraining Distribution:")
print(y_train.value_counts())

print("\nValidation Distribution:")
print(y_val.value_counts())

print("\nTesting Distribution:")
print(y_test.value_counts())

# Check time ranges
for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        data['timestamp'].min(),
        "to",
        data['timestamp'].max()
    )

Training: (149338, 10)
Validation: (42668, 10)
Testing: (21335, 10)

Training Distribution:
is_subscription
0    145091
1      4247
Name: count, dtype: int64

Validation Distribution:
is_subscription
0    40636
1     2032
Name: count, dtype: int64

Testing Distribution:
is_subscription
0    20270
1     1065
Name: count, dtype: int64
Train 2023-10-15 01:16:58 to 2025-11-16 13:29:21
Validation 2025-11-16 13:29:26 to 2026-06-28 10:07:43
Test 2026-06-28 10:21:06 to 2026-10-14 23:09:35


## Performing Random Forest 

In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=3,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

# Train Model
rf.fit(X_train, y_train)

# Validation Prediction
y_val_pred = rf.predict(X_val)
y_val_prob = rf.predict_proba(X_val)[:, 1]

# Evaluation
print("Random Forest Validation Results")

print("\nClassification Report:")
print(classification_report(y_val, y_val_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("\nAverage Precision (PR-AUC Summary):")
print(round(average_precision_score(y_val, y_val_prob), 4))

Random Forest Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9895    0.9968    0.9932     40636
           1     0.9256    0.7894    0.8521      2032

    accuracy                         0.9869     42668
   macro avg     0.9576    0.8931    0.9226     42668
weighted avg     0.9865    0.9869    0.9865     42668


Confusion Matrix:
[[40507   129]
 [  428  1604]]

Average Precision (PR-AUC Summary):
0.9276


## Performing XGBoost Classifier

In [24]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Handle class imbalance
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost Model
xgb = XGBClassifier(
    n_estimators=350,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=5,
    scale_pos_weight=scale_weight,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Training
xgb.fit(X_train, y_train)

# Validation Prediction
y_val_pred_xgb = xgb.predict(X_val)
y_val_prob_xgb = xgb.predict_proba(X_val)[:, 1]

# Evaluation
print("XGBoost Validation Results")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_xgb, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_xgb))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, y_val_prob_xgb), 4
))

XGBoost Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9942    0.9921    0.9932     40636
           1     0.8484    0.8844    0.8660      2032

    accuracy                         0.9870     42668
   macro avg     0.9213    0.9382    0.9296     42668
weighted avg     0.9873    0.9870    0.9871     42668


Confusion Matrix:
[[40315   321]
 [  235  1797]]

Average Precision:
0.9419


## Performing LightGBM

In [25]:
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Class Imbalance Ratio
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# LightGBM Model
lgbm = LGBMClassifier(
    n_estimators=350,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Model Training
lgbm.fit(X_train, y_train)

# Validation Predictions
y_val_pred_lgbm = lgbm.predict(X_val)
y_val_prob_lgbm = lgbm.predict_proba(X_val)[:, 1]

# Evaluation
print("LightGBM Validation Results")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_lgbm, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_lgbm))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, y_val_prob_lgbm), 4
))

LightGBM Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9938    0.9941    0.9940     40636
           1     0.8813    0.8770    0.8791      2032

    accuracy                         0.9885     42668
   macro avg     0.9376    0.9355    0.9366     42668
weighted avg     0.9885    0.9885    0.9885     42668


Confusion Matrix:
[[40396   240]
 [  250  1782]]

Average Precision:
0.9446


## Performing Catboost 

In [26]:
from catboost import CatBoostClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# CatBoost Model
cat = CatBoostClassifier(
    iterations=400,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=5,
    auto_class_weights='Balanced',
    loss_function='Logloss',
    random_seed=42,
    verbose=False,
    thread_count=-1
)

# Train Model
cat.fit(X_train, y_train)

# Validation Predictions
y_val_pred_cat = cat.predict(X_val)
y_val_prob_cat = cat.predict_proba(X_val)[:, 1]

# Evaluation
print("CatBoost Validation Results")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_cat, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_cat))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, y_val_prob_cat), 4
))

CatBoost Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9951    0.9885    0.9918     40636
           1     0.7967    0.9026    0.8463      2032

    accuracy                         0.9844     42668
   macro avg     0.8959    0.9455    0.9191     42668
weighted avg     0.9856    0.9844    0.9849     42668


Confusion Matrix:
[[40168   468]
 [  198  1834]]

Average Precision:
0.9437


## Performing Extra Tree 

In [27]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

# Extra Trees Model
et = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=18,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

# Training
et.fit(X_train, y_train)

# Validation Predictions
y_val_pred_et = et.predict(X_val)
y_val_prob_et = et.predict_proba(X_val)[:, 1]

# Evaluation
print("Extra Trees Validation Results")

print("\nClassification Report:")
print(classification_report(
    y_val, y_val_pred_et, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_et))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, y_val_prob_et), 4
))

Extra Trees Validation Results

Classification Report:
              precision    recall  f1-score   support

           0     0.9920    0.9870    0.9895     40636
           1     0.7637    0.8415    0.8007      2032

    accuracy                         0.9801     42668
   macro avg     0.8779    0.9143    0.8951     42668
weighted avg     0.9812    0.9801    0.9805     42668


Confusion Matrix:
[[40107   529]
 [  322  1710]]

Average Precision:
0.8721


## Decision Threshold Optimization

In [28]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

models = {
    "XGBoost": y_val_prob_xgb,
    "LightGBM": y_val_prob_lgbm,
    "CatBoost": y_val_prob_cat
}

results = []

for name, probabilities in models.items():

    for threshold in np.arange(0.05, 0.96, 0.01):

        predictions = (
            probabilities >= threshold
        ).astype(int)

        precision = precision_score(
            y_val, predictions, zero_division=0
        )

        recall = recall_score(
            y_val, predictions, zero_division=0
        )

        f1 = f1_score(
            y_val, predictions, zero_division=0
        )

        results.append({
            "Model": name,
            "Threshold": round(threshold, 2),
            "Precision": precision,
            "Recall": recall,
            "F1": f1
        })

results_df = pd.DataFrame(results)

best_results = (
    results_df
    .sort_values("F1", ascending=False)
    .drop_duplicates("Model")
    .sort_values("F1", ascending=False)
)

print(
    best_results.round(4).to_string(index=False)
)

   Model  Threshold  Precision  Recall     F1
LightGBM       0.72     0.9396  0.8420 0.8881
CatBoost       0.79     0.9252  0.8460 0.8838
 XGBoost       0.63     0.8923  0.8602 0.8760


## Minimal Hyperparameter Tuning For LightGBM

In [29]:
import numpy as np

from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

configs = [
    {"n_estimators": 250, "num_leaves": 31, "max_depth": -1},
    {"n_estimators": 450, "num_leaves": 31, "max_depth": -1},
    {"n_estimators": 350, "num_leaves": 15, "max_depth": 8}
]

# Current best model
best_model = lgbm
best_threshold = 0.72

best_f1 = f1_score(
    y_val,
    (y_val_prob_lgbm >= best_threshold).astype(int)
)

for i, config in enumerate(configs, 1):

    model = LGBMClassifier(
        **config,
        learning_rate=0.05,
        min_child_samples=20,
        scale_pos_weight=scale_weight,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    probabilities = model.predict_proba(X_val)[:, 1]

    for threshold in np.arange(0.40, 0.91, 0.02):

        predictions = (probabilities >= threshold).astype(int)

        score = f1_score(y_val, predictions)

        if score > best_f1:
            best_f1 = score
            best_model = model
            best_threshold = round(float(threshold), 2)

    print(f"Configuration {i} Completed")

print("\n===== BEST RESULT =====")
print("Best F1:", round(best_f1, 4))
print("Best Threshold:", best_threshold)
print("Best Model:", best_model.get_params())

Configuration 1 Completed
Configuration 2 Completed
Configuration 3 Completed

===== BEST RESULT =====
Best F1: 0.8881
Best Threshold: 0.72
Best Model: {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 0.85, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 350, 'n_jobs': -1, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 0.85, 'subsample_for_bin': 200000, 'subsample_freq': 1, 'scale_pos_weight': np.float64(34.163174005180124), 'verbosity': -1}


## Final Validation & Feature Importance

In [30]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score
)

import pandas as pd

# Best Model Validation
val_prob = best_model.predict_proba(X_val)[:, 1]
val_pred = (val_prob >= best_threshold).astype(int)

print("===== FINAL VALIDATION REPORT =====")

print(classification_report(
    y_val, val_pred, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

print("\nAverage Precision:")
print(round(
    average_precision_score(y_val, val_prob), 4
))

# Feature Importance
importance_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_model.feature_importances_
})

importance_df = importance_df.sort_values(
    "Importance", ascending=False
)

print("\n===== FEATURE IMPORTANCE =====")
print(importance_df.to_string(index=False))

===== FINAL VALIDATION REPORT =====
              precision    recall  f1-score   support

           0     0.9921    0.9973    0.9947     40636
           1     0.9396    0.8420    0.8881      2032

    accuracy                         0.9899     42668
   macro avg     0.9659    0.9197    0.9414     42668
weighted avg     0.9896    0.9899    0.9896     42668


Confusion Matrix:
[[40526   110]
 [  321  1711]]

Average Precision:
0.9446

===== FEATURE IMPORTANCE =====
                Feature  Importance
                 amount        3534
      amount_difference        1880
days_since_last_payment        1537
      previous_interval        1311
   interval_consistency        1034
prior_merchant_payments         970
        monthly_pattern         141
         similar_amount          53
      quarterly_pattern          33
         yearly_pattern           7


## Final Test Evaluation

In [31]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# Final Test Prediction
test_prob = best_model.predict_proba(X_test)[:, 1]

test_pred = (test_prob >= best_threshold).astype(int)

print("===== FINAL TEST RESULTS =====")

print("\nClassification Report:")
print(classification_report(
    y_test, test_pred, digits=4
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\n===== FINAL METRICS =====")

print("Accuracy:", round(accuracy_score(y_test, test_pred), 4))

print("Precision:", round(precision_score(y_test, test_pred), 4))

print("Recall:", round(recall_score(y_test, test_pred), 4))

print("F1 Score:", round(f1_score(y_test, test_pred), 4))

print("Average Precision:", round(
    average_precision_score(y_test, test_prob), 4
))

print("ROC AUC:", round(
    roc_auc_score(y_test, test_prob), 4
))

===== FINAL TEST RESULTS =====

Classification Report:
              precision    recall  f1-score   support

           0     0.9947    0.9959    0.9953     20270
           1     0.9202    0.8986    0.9093      1065

    accuracy                         0.9910     21335
   macro avg     0.9574    0.9472    0.9523     21335
weighted avg     0.9910    0.9910    0.9910     21335


Confusion Matrix:
[[20187    83]
 [  108   957]]

===== FINAL METRICS =====
Accuracy: 0.991
Precision: 0.9202
Recall: 0.8986
F1 Score: 0.9093
Average Precision: 0.9547
ROC AUC: 0.9937


## All ML Models Performance Table

In [32]:

models = {
    "Random Forest": rf,
    "XGBoost": xgb,
    "LightGBM": lgbm,
    "CatBoost": cat,
    "Extra Trees": et
}

results = []

for name, model in models.items():

    y_pred = model.predict(X_val)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision": precision_score(y_val, y_pred),
        "Recall": recall_score(y_val, y_pred),
        "F1 Score": f1_score(y_val, y_pred)
    })

comparison_df = pd.DataFrame(results)

comparison_df = comparison_df.sort_values(
    by="F1 Score",
    ascending=False
)

display(comparison_df.round(4))

,Model,Accuracy,Precision,Recall,F1 Score
2,LightGBM,0.9885,0.8813,0.8770,0.8791
1,XGBoost,0.9870,0.8484,0.8844,0.8660
0,Random Forest,0.9869,0.9256,0.7894,0.8521
3,CatBoost,0.9844,0.7967,0.9026,0.8463
4,Extra Trees,0.9801,0.7637,0.8415,0.8007


## Save the model though the pickle

In [34]:
import pickle

model_package = {
    "model": best_model,
    "features": features,
    "threshold": best_threshold
}

with open("finova_subscription_model.pkl", "wb") as file:
    pickle.dump(model_package, file)

print("Model Saved Successfully!")

Model Saved Successfully!


## Verify Pickle Model

In [35]:
with open("finova_subscription_model.pkl", "rb") as file:
    saved_model = pickle.load(file)

model = saved_model["model"]
threshold = saved_model["threshold"]

predictions = model.predict_proba(
    X_test[saved_model["features"]].head(5)
)[:, 1]

print("Probabilities:", predictions)
print("Predictions:", (predictions >= threshold).astype(int))

Probabilities: [9.38102871e-05 7.84491013e-04 5.10034578e-05 7.12162645e-07
 1.34428183e-06]
Predictions: [0 0 0 0 0]


In [36]:
print("Subscription Frequency:")
print(sub['frequency_months'].value_counts())

print("\nSubscription Status:")
print(sub['status'].value_counts())

print("\nSubscription Amount:")
print(sub['amount'].describe())

print("\nBilling Day:")
print(sub['billing_day'].describe())

Subscription Frequency:
frequency_months
1    639
3     35
Name: count, dtype: int64

Subscription Status:
status
active     572
churned    102
Name: count, dtype: int64

Subscription Amount:
count    674.000000
mean      68.735727
std       75.526181
min        2.990000
25%       16.062500
50%       59.350000
75%       86.195000
max      749.460000
Name: amount, dtype: float64

Billing Day:
count    674.000000
mean      14.551929
std        8.099739
min        1.000000
25%        8.000000
50%       14.000000
75%       21.000000
max       28.000000
Name: billing_day, dtype: float64


## Next Billing Date Prediction

In [37]:
import pandas as pd

sub['start_date'] = pd.to_datetime(sub['start_date'])
sub['end_date'] = pd.to_datetime(sub['end_date'])

today = pd.Timestamp("2026-10-14")

def next_billing(row):

    if row['status'] != 'active':
        return pd.NaT

    if row['start_date'] > today:
        return pd.NaT

    if pd.notna(row['end_date']) and row['end_date'] <= today:
        return pd.NaT

    frequency = int(row['frequency_months'])
    billing_day = int(row['billing_day'])

    billing_month = row['start_date'].replace(day=1)

    next_date = billing_month + pd.Timedelta(
        days=billing_day - 1
    )

    while next_date <= today or next_date < row['start_date']:
        billing_month += pd.DateOffset(months=frequency)

        next_date = billing_month + pd.Timedelta(
            days=billing_day - 1
        )

    if pd.notna(row['end_date']) and next_date > row['end_date']:
        return pd.NaT

    return next_date


sub['next_billing_date'] = sub.apply(
    next_billing, axis=1
)

# Upcoming payments
upcoming = sub[
    sub['next_billing_date'].notna()
].sort_values('next_billing_date')

print("Total Upcoming Subscriptions:", len(upcoming))

print(upcoming[
    [
        'merchant_name',
        'amount',
        'frequency_months',
        'next_billing_date'
    ]
].head(15).to_string(index=False))

Total Upcoming Subscriptions: 564
  merchant_name  amount  frequency_months next_billing_date
   Amazon Prime   14.99                 1        2026-10-15
     State Farm   18.08                 1        2026-10-15
City Water Dept   36.48                 3        2026-10-15
        Spotify   11.99                 1        2026-10-15
        Xfinity   63.59                 1        2026-10-15
     State Farm   36.38                 1        2026-10-15
        Disney+   13.99                 1        2026-10-15
     Blue Cross  208.07                 1        2026-10-15
        Verizon   83.15                 1        2026-10-15
           AT&T   66.02                 1        2026-10-15
        Verizon   89.46                 1        2026-10-15
        Spotify   11.99                 1        2026-10-15
     State Farm   26.67                 1        2026-10-15
  National Grid   95.35                 1        2026-10-15
           AT&T   61.42                 1        2026-10-15


In [38]:
# Select subscriptions eligible for future billing
active_sub = sub[
    sub['next_billing_date'].notna()
].copy()

# Calculate Monthly Cost
active_sub['monthly_cost'] = (
    active_sub['amount'].abs() /
    active_sub['frequency_months']
)

# Calculate Yearly Cost
active_sub['yearly_cost'] = (
    active_sub['monthly_cost'] * 12
)

# Customer-wise Summary
customer_summary = active_sub.groupby(
    'customer_id'
).agg(
    Active_Subscriptions=('subscription_id', 'count'),
    Monthly_Cost=('monthly_cost', 'sum'),
    Yearly_Cost=('yearly_cost', 'sum')
).reset_index()

# Round values
customer_summary[
    ['Monthly_Cost', 'Yearly_Cost']
] = customer_summary[
    ['Monthly_Cost', 'Yearly_Cost']
].round(2)

# Display results
print("Total Customers:", len(customer_summary))

print("\nTotal Monthly Subscription Cost:")
print(round(active_sub['monthly_cost'].sum(), 2))

print("\nTotal Yearly Subscription Cost:")
print(round(active_sub['yearly_cost'].sum(), 2))

print("\nCustomer-wise Subscription Summary:")
display(customer_summary.head(10))

Total Customers: 98

Total Monthly Subscription Cost:
34191.08

Total Yearly Subscription Cost:
410292.96

Customer-wise Subscription Summary:


,customer_id,Active_Subscriptions,Monthly_Cost,Yearly_Cost
0,C0000001,6,245.23,2942.80
1,C0000002,9,399.72,4796.64
2,C0000003,3,155.71,1868.48
3,C0000004,4,239.66,2875.92
4,C0000005,9,523.24,6278.88
5,C0000006,3,86.97,1043.64
6,C0000007,2,89.40,1072.80
7,C0000008,5,289.77,3477.20
8,C0000009,4,288.20,3458.40
9,C0000010,7,718.70,8624.40


## Upcoming Subscription Payment Alerts

In [39]:
# Reference Date
today = pd.Timestamp("2026-10-14")

# Calculate days until payment
active_sub['days_until_payment'] = (
    active_sub['next_billing_date'] - today
).dt.days

# Upcoming Alerts
for days in [3, 7, 30]:

    alerts = active_sub[
        active_sub['days_until_payment'].between(0, days)
    ].copy()

    print(f"\n===== NEXT {days} DAYS ALERT =====")

    print("Upcoming Payments:", len(alerts))

    print("Total Payment Amount:",
          round(alerts['amount'].abs().sum(), 2))

    print("\nUpcoming Subscription Details:")

    display(
        alerts[
            [
                'customer_id',
                'merchant_name',
                'amount',
                'next_billing_date',
                'days_until_payment'
            ]
        ].sort_values(
            'next_billing_date'
        ).head(10)
    )


===== NEXT 3 DAYS ALERT =====
Upcoming Payments: 54
Total Payment Amount: 3396.23

Upcoming Subscription Details:


,customer_id,merchant_name,amount,next_billing_date,days_until_payment
3,C0000001,Spotify,11.99,2026-10-15,1
406,C0000062,Disney+,13.99,2026-10-15,1
337,C0000052,Verizon,83.15,2026-10-15,1
421,C0000065,AT&T,66.02,2026-10-15,1
432,C0000067,Spotify,11.99,2026-10-15,1
276,C0000043,Blue Cross,208.07,2026-10-15,1
495,C0000076,National Grid,95.35,2026-10-15,1
392,C0000060,Amazon Prime,14.99,2026-10-15,1
85,C0000013,Hulu,17.99,2026-10-15,1
538,C0000082,State Farm,18.08,2026-10-15,1



===== NEXT 7 DAYS ALERT =====
Upcoming Payments: 139
Total Payment Amount: 9257.72

Upcoming Subscription Details:


,customer_id,merchant_name,amount,next_billing_date,days_until_payment
3,C0000001,Spotify,11.99,2026-10-15,1
276,C0000043,Blue Cross,208.07,2026-10-15,1
337,C0000052,Verizon,83.15,2026-10-15,1
392,C0000060,Amazon Prime,14.99,2026-10-15,1
406,C0000062,Disney+,13.99,2026-10-15,1
421,C0000065,AT&T,66.02,2026-10-15,1
90,C0000013,Spotify,11.99,2026-10-15,1
85,C0000013,Hulu,17.99,2026-10-15,1
432,C0000067,Spotify,11.99,2026-10-15,1
495,C0000076,National Grid,95.35,2026-10-15,1



===== NEXT 30 DAYS ALERT =====
Upcoming Payments: 521
Total Payment Amount: 33010.7

Upcoming Subscription Details:


,customer_id,merchant_name,amount,next_billing_date,days_until_payment
392,C0000060,Amazon Prime,14.99,2026-10-15,1
495,C0000076,National Grid,95.35,2026-10-15,1
90,C0000013,Spotify,11.99,2026-10-15,1
71,C0000011,State Farm,36.38,2026-10-15,1
538,C0000082,State Farm,18.08,2026-10-15,1
432,C0000067,Spotify,11.99,2026-10-15,1
53,C0000008,AT&T,61.42,2026-10-15,1
421,C0000065,AT&T,66.02,2026-10-15,1
575,C0000087,Verizon,89.46,2026-10-15,1
406,C0000062,Disney+,13.99,2026-10-15,1


## Subscription Price Increase Detection

In [40]:
# Select subscription transactions
price_df = sd[
    (sd['is_subscription'] == True) &
    (sd['amount'] < 0)
].copy()

# Convert timestamp
price_df['timestamp'] = pd.to_datetime(
    price_df['timestamp']
)

# Positive payment amount
price_df['current_amount'] = price_df['amount'].abs()

# Sort transactions
price_df = price_df.sort_values([
    'customer_id',
    'account_id',
    'merchant_name',
    'timestamp'
])

# Group by customer and merchant
group = price_df.groupby([
    'customer_id',
    'account_id',
    'merchant_name'
])

# Previous payment amount
price_df['previous_amount'] = (
    group['current_amount'].shift(1)
)

# Amount increase
price_df['amount_increase'] = (
    price_df['current_amount'] -
    price_df['previous_amount']
)

# Percentage increase
price_df['increase_percentage'] = (
    price_df['amount_increase'] /
    price_df['previous_amount'] * 100
)

# Detect significant increases
price_increases = price_df[
    (price_df['amount_increase'] >= 1) &
    (price_df['increase_percentage'] >= 5)
].copy()

print("Total Price Increase Events:", len(price_increases))

display(
    price_increases[[
        'customer_id',
        'merchant_name',
        'previous_amount',
        'current_amount',
        'amount_increase',
        'increase_percentage'
    ]].head(15).round(2)
)

Total Price Increase Events: 5


,customer_id,merchant_name,previous_amount,current_amount,amount_increase,increase_percentage
53754,C0000024,Verizon,98.46,103.65,5.19,5.27
64326,C0000029,Xfinity,75.09,79.05,3.96,5.27
100506,C0000046,State Farm,22.78,24.05,1.27,5.58
101409,C0000047,State Farm,26.60,28.00,1.40,5.26
101701,C0000047,Xfinity,76.65,80.69,4.04,5.27


## Customer-Level Price Increase Alert

In [41]:
# Select a customer
customer_id = "C0000047"

# Customer-specific price increases
customer_alerts = price_increases[
    price_increases['customer_id'] == customer_id
].copy()

# Sort latest events first
customer_alerts = customer_alerts.sort_values(
    'timestamp', ascending=False
)

print("Customer:", customer_id)
print("Total Price Increase Alerts:", len(customer_alerts))

display(
    customer_alerts[[
        'merchant_name',
        'previous_amount',
        'current_amount',
        'amount_increase',
        'increase_percentage',
        'timestamp'
    ]].round(2)
)

Customer: C0000047
Total Price Increase Alerts: 2


,merchant_name,previous_amount,current_amount,amount_increase,increase_percentage,timestamp
101701,Xfinity,76.65,80.69,4.04,5.27,2026-01-08 18:40:19
101409,State Farm,26.60,28.00,1.40,5.26,2026-01-08 08:35:06


## Complete Customer Subscription Report

In [42]:
customer_id = "C0000047"

# Customer subscriptions
customer_subs = active_sub[
    active_sub['customer_id'] == customer_id
].copy()

print("===== FINOVA AI SUBSCRIPTION REPORT =====")
print("Customer:", customer_id)

print("\nActive Subscriptions:", len(customer_subs))

print("Monthly Cost: $",
      round(customer_subs['monthly_cost'].sum(), 2))

print("Yearly Cost: $",
      round(customer_subs['yearly_cost'].sum(), 2))

# Active subscriptions
print("\n===== ACTIVE SUBSCRIPTIONS =====")

display(customer_subs[[
    'merchant_name',
    'amount',
    'frequency_months',
    'next_billing_date'
]])

# Upcoming payments
upcoming_30 = customer_subs[
    customer_subs['days_until_payment'].between(0, 30)
]

print("\n===== UPCOMING PAYMENTS (30 DAYS) =====")

display(upcoming_30[[
    'merchant_name',
    'amount',
    'next_billing_date',
    'days_until_payment'
]])

# Historical price increases
customer_prices = price_increases[
    price_increases['customer_id'] == customer_id
]

print("\n===== PRICE INCREASE HISTORY =====")

display(customer_prices[[
    'merchant_name',
    'previous_amount',
    'current_amount',
    'increase_percentage'
]].round(2))

===== FINOVA AI SUBSCRIPTION REPORT =====
Customer: C0000047

Active Subscriptions: 5
Monthly Cost: $ 459.76
Yearly Cost: $ 5517.12

===== ACTIVE SUBSCRIPTIONS =====


,merchant_name,amount,frequency_months,next_billing_date
295,Xfinity,71.56,1,2026-11-08
296,State Farm,24.83,1,2026-11-08
297,Geico Auto,141.66,1,2026-10-24
298,Amazon Prime,14.99,1,2026-10-25
300,Blue Cross,206.72,1,2026-10-19



===== UPCOMING PAYMENTS (30 DAYS) =====


,merchant_name,amount,next_billing_date,days_until_payment
295,Xfinity,71.56,2026-11-08,25
296,State Farm,24.83,2026-11-08,25
297,Geico Auto,141.66,2026-10-24,10
298,Amazon Prime,14.99,2026-10-25,11
300,Blue Cross,206.72,2026-10-19,5



===== PRICE INCREASE HISTORY =====


,merchant_name,previous_amount,current_amount,increase_percentage
101409,State Farm,26.60,28.00,5.26
101701,Xfinity,76.65,80.69,5.27


## Model-Based Subscription Detection

In [44]:
# Load saved model
with open("finova_subscription_model.pkl", "rb") as f:
    saved = pickle.load(f)

model = saved["model"]
features = saved["features"]
threshold = saved["threshold"]

# Select customer
customer_id = "C0000047"

customer_tx = sd[
    sd["customer_id"] == customer_id
].copy()

# ML Prediction
customer_tx["subscription_probability"] = (
    model.predict_proba(customer_tx[features])[:, 1]
)

customer_tx["predicted_subscription"] = (
    customer_tx["subscription_probability"] >= threshold
)

# Detected subscription transactions
detected = customer_tx[
    customer_tx["predicted_subscription"]
].copy()

print("Customer:", customer_id)
print("Total Transactions:", len(customer_tx))
print("ML Detected Subscription Transactions:", len(detected))

print("\nDetected Merchants:")

print(
    detected["merchant_name"].value_counts()
)

print("\nDetected Transactions:")

display(
    detected[
        [
            "timestamp",
            "merchant_name",
            "amount",
            "subscription_probability"
        ]
    ].sort_values(
        "timestamp", ascending=False
    ).head(15)
)

Customer: C0000047
Total Transactions: 915
ML Detected Subscription Transactions: 3

Detected Merchants:
merchant_name
Realty Partners    2
Netflix            1
Name: count, dtype: int64

Detected Transactions:


,timestamp,merchant_name,amount,subscription_probability
101315,2026-10-02 14:09:07,Realty Partners,-731.74,0.741148
101314,2026-09-02 13:32:28,Realty Partners,-731.74,0.740539
101268,2025-10-04 17:40:04,Netflix,-16.60,0.724187


In [45]:
# Actual Subscription Labels
actual = customer_tx['is_subscription'].astype(bool)

# ML Predictions
predicted = customer_tx['predicted_subscription'].astype(bool)

print("===== CUSTOMER MODEL ANALYSIS =====")

print("Actual Subscription Transactions:", actual.sum())

print("ML Detected Transactions:", predicted.sum())

print("Correctly Detected:", (actual & predicted).sum())

print("Missed Subscriptions:", (actual & ~predicted).sum())

print("False Positives:", (~actual & predicted).sum())

# Merchant-wise Analysis
actual_subs = customer_tx[actual].copy()

merchant_report = actual_subs.groupby('merchant_name').agg(
    Actual_Payments=('transaction_id', 'count'),
    Detected_Payments=('predicted_subscription', 'sum'),
    Average_Probability=('subscription_probability', 'mean'),
    Maximum_Probability=('subscription_probability', 'max')
).reset_index()

merchant_report = merchant_report.sort_values(
    'Actual_Payments',
    ascending=False
)

print("\n===== MERCHANT-WISE DETECTION =====")

display(merchant_report.round(4))

===== CUSTOMER MODEL ANALYSIS =====
Actual Subscription Transactions: 12
ML Detected Transactions: 3
Correctly Detected: 1
Missed Subscriptions: 11
False Positives: 2

===== MERCHANT-WISE DETECTION =====


,merchant_name,Actual_Payments,Detected_Payments,Average_Probability,Maximum_Probability
0,Netflix,8,1,0.4687,0.7242
1,State Farm,2,0,0.0006,0.0010
2,Xfinity,2,0,0.0006,0.0010


In [46]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

customer_id = "C0000047"

# Customer transactions
customer_tx = sd[
    sd['customer_id'] == customer_id
].copy()

# Prepare features exactly like training
X_customer = customer_tx[features].copy()

# Fix amount
X_customer['amount'] = X_customer['amount'].abs()

# Fix boolean features
bool_cols = [
    'monthly_pattern',
    'similar_amount',
    'quarterly_pattern',
    'yearly_pattern'
]

X_customer[bool_cols] = X_customer[bool_cols].astype(int)

# Handle missing values
missing_cols = [
    'days_since_last_payment',
    'amount_difference',
    'previous_interval',
    'interval_consistency'
]

X_customer[missing_cols] = X_customer[missing_cols].fillna(-1)

# Predict
customer_tx['subscription_probability'] = (
    best_model.predict_proba(X_customer)[:, 1]
)

customer_tx['predicted_subscription'] = (
    customer_tx['subscription_probability'] >= best_threshold
)

# Evaluation
actual = customer_tx['is_subscription'].astype(int)
predicted = customer_tx['predicted_subscription'].astype(int)

print("===== CORRECTED CUSTOMER RESULTS =====")

print("Actual Subscriptions:", actual.sum())
print("Detected Subscriptions:", predicted.sum())

print("\nClassification Report:")
print(classification_report(
    actual, predicted, digits=4, zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(actual, predicted))

===== CORRECTED CUSTOMER RESULTS =====
Actual Subscriptions: 12
Detected Subscriptions: 34

Classification Report:
              precision    recall  f1-score   support

           0     0.9955    0.9712    0.9832       903
           1     0.2353    0.6667    0.3478        12

    accuracy                         0.9672       915
   macro avg     0.6154    0.8189    0.6655       915
weighted avg     0.9855    0.9672    0.9749       915


Confusion Matrix:
[[877  26]
 [  4   8]]


In [47]:
# False Positive Transactions
fp = customer_tx[
    (customer_tx['is_subscription'] == False) &
    (customer_tx['predicted_subscription'] == True)
].copy()

# False Negative Transactions
fn = customer_tx[
    (customer_tx['is_subscription'] == True) &
    (customer_tx['predicted_subscription'] == False)
].copy()

print("===== FALSE POSITIVE ANALYSIS =====")

print("Total False Positives:", len(fp))

print("\nFalse Positive Merchants:")
print(fp['merchant_name'].value_counts())

print("\nFalse Positive Details:")
display(
    fp[[
        'merchant_name',
        'amount',
        'days_since_last_payment',
        'amount_difference',
        'interval_consistency',
        'subscription_probability'
    ]].round(4)
)

print("\n===== FALSE NEGATIVE ANALYSIS =====")

print("Total False Negatives:", len(fn))

print("\nMissed Subscription Merchants:")
print(fn['merchant_name'].value_counts())

print("\nFalse Negative Details:")
display(
    fn[[
        'merchant_name',
        'amount',
        'days_since_last_payment',
        'amount_difference',
        'interval_consistency',
        'subscription_probability'
    ]].round(4)
)

===== FALSE POSITIVE ANALYSIS =====
Total False Positives: 26

False Positive Merchants:
merchant_name
Vela Neobank - Overdraft Fee    3
Apple iCloud+                   2
ATM                             2
Shell                           2
AMC Theatres                    1
Booking.com                     1
Amazon Prime                    1
Dental Associates               1
Family Clinic                   1
LensCrafters                    1
Lowe's                          1
Home Depot                      1
Kroger                          1
Macy's                          1
MTA Transit                     1
Nike                            1
Marriott                        1
Realty Partners                 1
Starbucks                       1
Trader Joe's                    1
Walmart                         1
Name: count, dtype: int64

False Positive Details:


,merchant_name,amount,days_since_last_payment,amount_difference,interval_consistency,subscription_probability
100821,AMC Theatres,-16.00,599.6771,9.73,411.4782,0.7987
100823,ATM,-60.00,NaN,NaN,NaN,0.7686
100834,ATM,-60.00,28.9889,0.00,0.9131,0.9989
100886,Amazon Prime,-16.43,NaN,NaN,NaN,0.7363
100909,Apple iCloud+,-3.27,NaN,NaN,NaN,0.7234
100910,Apple iCloud+,-3.26,31.5127,0.01,NaN,0.7940
100922,Booking.com,-89.50,NaN,NaN,NaN,0.8810
101013,Dental Associates,-84.30,NaN,NaN,NaN,0.8692
101097,Family Clinic,-77.20,NaN,NaN,NaN,0.8640
101119,Home Depot,-93.21,NaN,NaN,NaN,0.8608



===== FALSE NEGATIVE ANALYSIS =====
Total False Negatives: 4

Missed Subscription Merchants:
merchant_name
State Farm    2
Netflix       1
Xfinity       1
Name: count, dtype: int64

False Negative Details:


,merchant_name,amount,days_since_last_payment,amount_difference,interval_consistency,subscription_probability
101264,Netflix,-16.60,66.2623,0.24,NaN,0.6576
101408,State Farm,-26.60,NaN,NaN,NaN,0.4627
101409,State Farm,-28.00,457.0075,1.40,NaN,0.3168
101701,Xfinity,-80.69,457.3680,4.04,NaN,0.4313


In [48]:
import numpy as np
import pandas as pd

# ML predicted transactions
candidates = customer_tx[
    customer_tx['predicted_subscription'] == True
].copy()

# Historical payment evidence
candidates['has_history'] = (
    candidates['prior_merchant_payments'] >= 2
)

candidates['stable_amount'] = (
    candidates['amount_difference'] <= 2
)

candidates['regular_interval'] = (
    candidates['days_since_last_payment'].between(27, 32) |
    candidates['days_since_last_payment'].between(85, 95) |
    candidates['days_since_last_payment'].between(350, 380)
)

candidates['stable_timing'] = (
    candidates['interval_consistency'] <= 5
)

# Strong recurring evidence
candidates['recurring_evidence'] = (
    candidates['has_history'] &
    candidates['stable_amount'] &
    candidates['regular_interval'] &
    candidates['stable_timing']
)

# Assign review status
candidates['status'] = np.where(
    candidates['recurring_evidence'],
    'Recurring Candidate',
    'Needs Review'
)

print("===== HYBRID DETECTION ANALYSIS =====")

print("\nTotal ML Candidates:", len(candidates))

print("\nStatus Distribution:")
print(candidates['status'].value_counts())

print("\nMerchant-wise Results:")

display(
    candidates[
        [
            'merchant_name',
            'subscription_probability',
            'prior_merchant_payments',
            'days_since_last_payment',
            'amount_difference',
            'status'
        ]
    ].round(3)
)

===== HYBRID DETECTION ANALYSIS =====

Total ML Candidates: 34

Status Distribution:
status
Needs Review           26
Recurring Candidate     8
Name: count, dtype: int64

Merchant-wise Results:


,merchant_name,subscription_probability,prior_merchant_payments,days_since_last_payment,amount_difference,status
100821,AMC Theatres,0.799,2,599.677,9.73,Needs Review
100823,ATM,0.769,0,NaN,NaN,Needs Review
100834,ATM,0.999,11,28.989,0.00,Recurring Candidate
100886,Amazon Prime,0.736,0,NaN,NaN,Needs Review
100909,Apple iCloud+,0.723,0,NaN,NaN,Needs Review
100910,Apple iCloud+,0.794,1,31.513,0.01,Needs Review
100922,Booking.com,0.881,0,NaN,NaN,Needs Review
101013,Dental Associates,0.869,0,NaN,NaN,Needs Review
101097,Family Clinic,0.864,0,NaN,NaN,Needs Review
101119,Home Depot,0.861,0,NaN,NaN,Needs Review


In [ ]:
# Copy ML detected candidates
audit = candidates.copy()

merchant = audit['merchant_name'].fillna('').str.lower()

# Payment category identification
conditions = [
    merchant.eq('atm') |
    audit['is_cash_withdrawal'].fillna(False).astype(bool),

    merchant.str.contains(
        r'overdraft|bank fee|service fee', regex=True
    ),

    merchant.str.contains(
        r'realty|rent|landlord', regex=True
    ),

    merchant.str.contains(
        r'netflix|spotify|amazon prime|icloud|disney\+|hulu|youtube|adobe',
        regex=True
    ),

    merchant.str.contains(
        r'xfinity|state farm|verizon|at&t|t-mobile|conedison|national grid',
        regex=True
    )
]

categories = [
    'Cash Withdrawal',
    'Banking Fee',
    'Housing Payment',
    'Digital Subscription',
    'Recurring Bill'
]

audit['payment_type'] = np.select(
    conditions,
    categories,
    default='Needs Category Review'
)

# Merchant-level summary
merchant_summary = audit.groupby(
    ['merchant_name', 'payment_type']
).agg(
    ML_Detections=('transaction_id', 'count'),
    Strong_Recurring=('recurring_evidence', 'sum'),
    Maximum_Probability=('subscription_probability', 'max')
).reset_index()

# Dashboard decisions
def decide(row):
    category = row['payment_type']
    recurring = row['Strong_Recurring']

    if category in ['Cash Withdrawal', 'Banking Fee']:
        return 'Exclude from Subscriptions'

    if category == 'Housing Payment':
        return 'Track as Housing'

    if recurring >= 1 and category == 'Digital Subscription':
        return 'Subscription Candidate'

    if recurring >= 1 and category == 'Recurring Bill':
        return 'Recurring Bill Candidate'

    return 'Needs Review'

merchant_summary['Decision'] = merchant_summary.apply(
    decide, axis=1
)

print("===== PAYMENT INTELLIGENCE =====")

display(
    merchant_summary.sort_values(
        'Strong_Recurring', ascending=False
    ).round(3)
)